# 🤖 Week 6 — AI in Python
**Carojon Academy · Python, AI & the Web**

> **Today's goal:** understand **how AI really works** (no magic!), learn to **prompt** well, and build your own **AI study-buddy chatbot**.

## 🗺️ You are here: Week 6 of 8 (the big one!)
`1 ✅` → `2 ✅` → `3 ✅` → `4 ✅` → `5 ✅` → **`6 AI in Python`** → 7 HTML/CSS → 8 JavaScript + Capstone

## 🔁 Warm-up
1. What does an API do?
2. What's inside a JSON reply?
3. Where do we store API keys?
<details><summary>Answers</summary>Lets programs talk · dictionaries and lists · environment variables</details>

In [ ]:
from helpers import *
import gradio as gr
import random

## 1️⃣ What is AI, really?

**Analogy:** Imagine a student who has **read millions of books and websites** and is brilliant at one game: **"what word comes next?"**

> "The cat sat on the ___" → "mat"

A **Large Language Model (LLM)**, like Claude, is that game played at *enormous* scale:
- It reads text in small pieces called **tokens** (roughly word-chunks)
- It learned **patterns** from huge amounts of training text
- To answer you, it predicts the next token, then the next, then the next, very fast

It's **not magic**, and it's **not a person**. It's pattern-prediction that got *so* good it can write, explain and code.

## 2️⃣ Build a tiny language model yourself 🧪

We'll make a **next-word predictor**. It looks at what words followed each word in some training text, then picks a follower at random.

(It's the *same idea* as the word counter from Week 4.)

In [ ]:
# 🎛️ TUTOR LIVE-TWEAK: change the training text and watch the "AI" change!
training_text = """
the cat sat on the mat the cat ate the fish the dog sat on the log
the dog ate the bone the cat chased the dog the dog chased the cat
"""

words = training_text.split()

# Step 1: learn which words follow which
following = {}
for current, nxt in zip(words, words[1:]):
    if current not in following:
        following[current] = []
    following[current].append(nxt)

print(following["the"])

In [ ]:
# Step 2: generate text by repeatedly predicting the next word
def generate(start, length=10):
    output = [start]
    for _ in range(length):
        options = following.get(output[-1])
        if not options:
            break
        output.append(random.choice(options))
    return " ".join(output)

for _ in range(3):
    print(generate("the"))

In [ ]:
demo = gr.Interface(
    fn=generate,
    inputs=[gr.Textbox(value="the", label="Start word"), gr.Slider(3, 30, value=10, step=1, label="How many words")],
    outputs=gr.Textbox(label="Generated text"),
    title="🔮 Tiny Language Model",
    flagging_mode="never",
)
show(demo)

### 💡 What did we just learn?
1. **More training text = better predictions.** Real models train on trillions of words.
2. **Look at `following["the"]`:** words that appear more often are chosen more often (probability!).
3. **Our model only looks 1 word back.** Real models look at thousands of words at once, so they can follow a whole conversation.
4. **The model can only say things that look like its training data.**

### ⚖️ Bias: garbage in, garbage out
Change `training_text` to **only sentences about football**. Now ask it to say something about cooking. 😅 If training data is unfair, narrow or wrong, the AI's answers are too. This is **bias**, a major real-world issue.

### 🤥 Hallucination
Because AI predicts *plausible* text, it can say things that **sound right but are false**, with total confidence. **Always check important facts.**

## 3️⃣ Prompting: how to talk to an AI

A **prompt** is the instruction you give. Better prompt → better answer.

**Analogy:** Giving directions to a taxi driver. "Take me somewhere nice" ❌ vs "Take me to Eke Awka market, use the main road" ✅.

### The R-T-C-F recipe 🍳

| Letter | What to include | Example |
| ------ | --------------- | ------- |
| **R**ole | Who should the AI be? | "You are a friendly maths tutor for a 12-year-old" |
| **T**ask | What exactly to do? | "Explain what a variable is" |
| **C**ontext | Useful background | "They know football but not coding" |
| **F**ormat | How to answer | "Use 3 short sentences and one analogy" |

**Weak:** "explain variables"
**Strong:** "You are a friendly coding tutor for a 12-year-old who loves football. Explain what a variable is using a football analogy, in 3 short sentences."

### ✏️ Prompt lab
Rewrite these weak prompts using R-T-C-F:
1. "Tell me about Nigeria."
2. "Help with my homework."
3. "Make a website."

(Use any AI chat app with your tutor. Compare weak vs strong answers.)

## 4️⃣ Build a chatbot (the old way: rules) 📜

Before AI, chatbots were **hand-written rules**: if the message contains a keyword, give a stored answer. Let's build one, then see its limits.

Gradio has a ready-made chat screen: `gr.ChatInterface`. Your function receives `(message, history)` and **returns the reply**.

In [ ]:
FAQ = {
    "hello": "Hi there! 👋 Ask me about variables, loops or functions.",
    "variable": "A variable is a labelled box that stores a value, like age = 12.",
    "loop": "A loop repeats code. Use `for` when you know how many times, `while` when you don't.",
    "function": "A function is a reusable block of code, like a blender: ingredients in, smoothie out.",
    "list": "A list holds many items in order. The first item has index 0.",
}   # 🎛️ Tutor: add more topics live

def rule_bot(message, history):
    text = message.lower()
    for keyword, answer in FAQ.items():
        if keyword in text:
            return answer
    return "Hmm, I don't know that yet 🤔. Try asking about: " + ", ".join(FAQ)

demo = gr.ChatInterface(fn=rule_bot, title="📜 Rule-Based Study Bot")
show(demo, height=560)

### 🧐 Limits of the rule bot
Try asking: *"Can you explain how to repeat something 10 times?"* It fails! It only knows exact keywords. **Real AI understands meaning**, not just keywords. That's the leap.

## 5️⃣ Upgrade: plug in a real AI brain 🧠

**Tutor demo.** We call an AI through its **API** (Week 5!). The tutor's secret key lives in an **environment variable**. Students never see it.

### Setup (tutor, once)
```bash
uv add anthropic
export ANTHROPIC_API_KEY="sk-ant-..."      # Mac/Linux
# Windows PowerShell:  $env:ANTHROPIC_API_KEY="sk-ant-..."
uv run jupyter lab
```

### The messages list (remember lists of dictionaries?)
```python
[{"role": "user", "content": "Hi"},
 {"role": "assistant", "content": "Hello! How can I help?"},
 {"role": "user", "content": "What is a loop?"}]
```

In [ ]:
import os

# 🎛️ TUTOR LIVE-TWEAK: change the personality and watch the bot change!
SYSTEM_PROMPT = """You are 'Coach Chuks', a friendly, encouraging coding tutor for 12 to 15 year olds in Nigeria.
Explain with short, simple answers and one fun everyday analogy (football, jollof rice, danfo buses, markets).
Never just give a full homework answer; guide with hints first.
If you are not sure, say so."""

MODEL = "claude-haiku-5-5"   # fast and cheap for classrooms; swap for "claude-sonnet-5-5" for smarter answers


def _to_text(content):
    """Gradio versions format chat content differently. This handles all of them."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    return str(content)


def _history_to_messages(history):
    messages = []
    for turn in history:
        if isinstance(turn, dict):                      # newer format
            messages.append({"role": turn["role"], "content": _to_text(turn["content"])})
        else:                                           # older format: (user, bot) pairs
            user, bot = turn
            messages.append({"role": "user", "content": _to_text(user)})
            if bot:
                messages.append({"role": "assistant", "content": _to_text(bot)})
    return messages


def ai_bot(message, history):
    if not os.getenv("ANTHROPIC_API_KEY"):
        return "(No AI key set, using the rule-based bot)\n\n" + rule_bot(message, history)
    try:
        from anthropic import Anthropic
        client = Anthropic()    # reads the key from the environment
        messages = _history_to_messages(history) + [{"role": "user", "content": message}]
        reply = client.messages.create(
            model=MODEL,
            max_tokens=400,
            system=SYSTEM_PROMPT,
            messages=messages,
        )
        return reply.content[0].text
    except Exception as error:
        return f"😢 The AI had a problem: {type(error).__name__}. Try again in a moment."


demo = gr.ChatInterface(fn=ai_bot, title="🤖 Coach Chuks: AI Study Buddy")
show(demo, height=560)

### 🔬 Experiments (do these as a class)
1. Change `SYSTEM_PROMPT` to make the bot a **pirate**, then back to a tutor. What changed?
2. Ask it something it *shouldn't* know (like tomorrow's lottery numbers). How does it respond?
3. Ask the **same question twice**. Are the answers identical? Why not? (Hint: our tiny model chose words *randomly* too.)
4. Ask it to write code for the `times_table` app from Week 2. **Read the code. Do you understand every line?**

## 6️⃣ Using AI wisely 🧭

| ✅ Do | ❌ Don't |
| ---- | ------- |
| Ask it to **explain** things in a different way | Copy answers without understanding |
| Ask for **hints**, then try yourself | Trust it blindly: it can be confidently wrong |
| **Test and read** any code it gives you | Paste passwords, API keys or private information into it |
| Use it to **check** your own work | Let it do the thinking you're here to learn |

**Your superpower:** AI is fast, but **you** decide what to build, what's right, and what's fair. People who understand code can direct AI and **catch its mistakes**. People who don't can't.

> 🧠 *AI is like a very fast intern who sometimes makes confident mistakes. You are the manager.*

## 🔨 Build along (with your tutor)
We design **your own AI persona app**: pick a topic you love (football coach, recipe helper, history storyteller...) and write the `SYSTEM_PROMPT` using R-T-C-F.

In [ ]:
# BUILD ALONG - our persona's SYSTEM_PROMPT and its Gradio app

## 🎯 Challenges

### ⭐ Starter (age 12)
Extend the **rule bot** with 5 new keywords and answers (e.g. `"string"`, `"dictionary"`, `"bug"`). Add a friendly emoji reply when someone says `"thanks"`.

### 🚀 Level up (age 15)
1. Improve `rule_bot` so it returns **all** matching answers if several keywords appear in one message.
2. Make the tiny language model **look 2 words back** instead of 1 (the dictionary key becomes a pair like `("the", "cat")`). Compare the quality of its sentences.
3. Write a one-page "AI Safety Rules" poster for your class: 5 rules for using AI honestly and safely.

## 🤖 The AI boom connection

You just did, in miniature, what AI engineers do:
- **Data → model** (our next-word table)
- **Prompt → response** (R-T-C-F, system prompts)
- **Model behind an API → app on top** (Gradio)

The jobs of the future are not just "AI builders". They're **AI-fluent doctors, farmers, designers, teachers, business owners**. Anyone who can combine **their own passion** + **coding** + **AI** will stand out.

## ✅ Recap quiz
1. In one sentence, how does an LLM produce text? <details><summary>Answer</summary>It predicts the next token over and over based on patterns learned from training data.</details>
2. What is a hallucination? <details><summary>Answer</summary>A confident answer that sounds right but is false.</details>
3. Why can biased training data be a problem? <details><summary>Answer</summary>The AI learns and repeats the unfairness or gaps in its data.</details>
4. What do R, T, C, F stand for? <details><summary>Answer</summary>Role, Task, Context, Format.</details>
5. Give two things you should never type into an AI chat. <details><summary>Answer</summary>Passwords/API keys and private personal information.</details>

## 🏠 Homework
Finish your **persona chatbot** (rule-based version for everyone; AI version if the tutor demos with their key). Write down **3 prompts that worked well** and **1 that failed**, and explain why.

## 🔜 Next two weeks: back to the Web!
You've built apps with Gradio, which creates web pages **for** you. In Weeks 7 and 8 you'll learn the three languages that build **every website in the world**: **HTML, CSS and JavaScript**, and use them to make your own page.